<div align="center">

# Week 03: Deep learning and convolutional neural networks

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lab.html) · [PDF notes](Week03_Lecture_Notes.pdf) · [Open in Colab (fallback)](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/NB03_deep_learning_and_cnns.ipynb)

</div>

**How to use this Jupyter notebook.** Run the cells from top to bottom with Shift+Enter. Section 0 prepares the environment and must run first. Exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells: Open them only after trying. *Observe and reflect* cells ask open questions and have no solution cell. *Python note* cells explain the language where a new construct first appears. A participant who opens the notebook in the middle runs the *Catch-up* cell at the start of that section.

**Three paths.** All cells without a label form the **core path**, which is enough for everybody, including those who have never written code. Cells labelled **Advanced path** are optional extensions for those who already program. The **prompt path** reaches the same results by writing prompts instead of code. Every exercise has a collapsed *Prompt route* with a ready prompt for an artificial intelligence (AI) assistant, and every section ends with a *Prompt challenge*. Copy the prompt into an assistant, replace the content of the exercise cell with the code it returns and run it, and let the check decide. The [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md) of the week collects all prompts, and the [guide to the prompt path](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/PROMPT_PATH.md) explains which assistants work with this notebook.

**Where it runs.** The notebook is written for Jupyter in Anaconda on your own computer (see SETUP.md in the course folder). If installing Anaconda is not possible, the same notebook opens in Google Colab through the link at the top. Training the networks of this notebook takes from a few seconds to about a minute per cell on an ordinary laptop, and no graphics card is needed. Every data set here is synthetic or comes from a public research database. Real patient data belongs only on your own computer, under the rules of your institution, and never in a cloud notebook or in an artificial intelligence (AI) chat.

## 0. Setup

The first cell finds the course folder. If the notebook was opened on its own, for example in Google Colab, it downloads the course package `vclinic` and the data files from GitHub. The second cell loads the packages, chooses the language of the messages and checks the optional packages of the week. PyTorch is the deep learning package of the course. Colab includes it; for Anaconda, SETUP.md shows the one-line installation.

In [ ]:
#@title Course files: run this cell (there is no need to open it)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
    "data/examples/echo/VC-0821.png",
    "data/examples/echo/VC-1873.png",
    "data/examples/echo/VC-1212.png",
    "models/week03/echo_cnn.pt",
    "models/week03/echo_cnn.json",
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # A downloaded copy, for example in Colab: the files of the package are refreshed.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Course files downloaded to", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Setup: packages, language and plotting style
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy
import math

from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from vclinic.imaging import CMR_CLASSES, ECHO_CLASSES, echo_images, make_cmr_set

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError as error:
    raise ImportError("PyTorch is not installed. Install it as SETUP.md describes (pip install torch), then restart the kernel.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 2, 0):
    raise ImportError("The course package vclinic is older than this notebook. Update the course folder (git pull, or download the ZIP file again) and restart the kernel.")
teach.set_language("en")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
print("vclinic version", vclinic.__version__)
print("Course folder:", ROOT)
for name in ["ipywidgets", "torchvision"]:
    try:
        importlib.import_module(name)
        print(f"{name}: installed")
    except ImportError:
        print(f"{name}: not installed (optional; the notebook uses a built-in fallback)")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
teach.register_answers({
    "Exercise 3.1": (49152, 0),
    "Exercise 3.2": (0.148, 0.001),
    "Exercise 3.3": (0.207, 0.001),
    "Exercise A": (0.859, 0.001),
    "Exercise B": (-0.5, 0.001),
    "Exercise C": (40, 0),
    "Exercise D": (31, 0),
    "Exercise E": (10098, 0),
    "Exercise F": (1.429, 0.001),
    "Exercise G": (0.916, 0.001),
    "Exercise H": (4640, 0),
    "Exercise I": (73984, 0),
    "Exercise J": (0.822, 0.02),
})
print("Expected answers are ready.")

## Python step 3: Images as arrays

For a computer, a grey-scale image is a table of numbers: one number per pixel, usually from 0 (black) to 1 (white). A stack of images is an array with three dimensions, and a colour image adds a fourth for the channels. This step builds an image by hand, reads one from a file, slices and normalises a stack of echocardiographic frames and, at the end, writes the training loop of a single neuron. Every later section of the notebook uses these steps.

In [ ]:
# Python practice: a small example for learning the language
img = np.zeros((7, 7))
img[1:6, 1:6] = 1.0      # a bright square
img[2:5, 2:5] = 0.2      # a dark centre, like a small ventricle
print(img.shape, img.dtype)
print(img)
plt.figure(figsize=(2.2, 2.2))
plt.imshow(img, cmap="gray", vmin=0, vmax=1)
plt.axis("off")
plt.show()

**Python note.** `np.zeros((7, 7))` makes an array of 7 rows and 7 columns filled with 0. `img[1:6, 1:6]` selects rows 1 to 5 and columns 1 to 5: In a slice `a:b`, the first number is included and the second is not. The first index is always the row, counted from the top, and the second the column. `plt.imshow` draws the array, and `vmin` and `vmax` fix which values are black and white.

In [ ]:
# Python practice: a small example for learning the language
frame = plt.imread(ROOT / "data" / "examples" / "echo" / "VC-1212.png")
print(frame.shape, frame.dtype, round(float(frame.min()), 3), round(float(frame.max()), 3))
plt.figure(figsize=(3, 3))
plt.imshow(frame, cmap="gray", vmin=0, vmax=1)
plt.title("VC-1212")
plt.axis("off")
plt.show()

**Python note.** `plt.imread` reads an image file into an array. A grey-scale image in the Portable Network Graphics (PNG) format gives a two-dimensional array with values from 0 to 1. A colour file would give an array of shape (rows, columns, 3) or (rows, columns, 4). `ROOT / "data" / ...` joins the parts of a path in the same way on Windows, macOS and Linux.

In [ ]:
# Python practice: a small example for learning the language
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))
X12, y12, table12 = echo_images(registry.head(12))
print(X12.shape, y12)
print(X12[0].shape, X12[:, 32, :].shape, round(float(X12[0, 32, 32]), 3))

**Python note.** `echo_images` draws one frame for every patient of the table it receives and returns three things, which the line unpacks into three names. `X12` has three dimensions: frame, row and column. `X12[0]` is the first frame, and `X12[:, 32, :]` takes row 32 of every frame. The colon alone means *all*.

In [ ]:
# Python practice: a small example for learning the language
x = X12[0]
z = (x - x.mean()) / x.std()                      # z-scores: mean 0, SD 1
m = (x - x.min()) / (x.max() - x.min())           # min-max: from 0 to 1
print(round(float(z.mean()), 3), round(float(z.std()), 3), round(float(m.min()), 3), round(float(m.max()), 3))
crop = x[16:48, 16:48]
print("crop", crop.shape)

**Python note.** Arithmetic on an array works on every element at once, so `x - x.mean()` subtracts the mean from all 4096 pixels. A network learns more easily from inputs near 0 with a spread near 1. The mean and the standard deviation (SD) must come from the training images only, as Section 3 shows. `x[16:48, 16:48]` crops the central 32 × 32 pixels.

In [ ]:
# Python practice: the training loop of one neuron
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


feats = registry[["lvedd_mm", "ivs_mm"]].to_numpy(float)
target = ((registry["lvedd_mm"] > 58) | (registry["ivs_mm"] >= 13)).astype(int).to_numpy()
Xs = (feats - feats.mean(axis=0)) / feats.std(axis=0)

w, b, lr = np.zeros(2), 0.0, 0.5
for step in range(201):
    p = sigmoid(Xs @ w + b)                                              # 1. forward pass
    loss = -np.mean(target * np.log(p) + (1 - target) * np.log(1 - p))  # 2. loss
    grad_w = Xs.T @ (p - target) / len(target)                           # 3. gradients
    grad_b = np.mean(p - target)
    if step % 50 == 0:
        print(step, round(loss, 4), np.round(w, 3), round(b, 3))
    w = w - lr * grad_w                                                  # 4. update
    b = b - lr * grad_b

**Python note.** Every training loop has the same four steps: the *forward pass* computes the predictions, the *loss* measures how wrong they are, the *gradients* say in which direction each weight should move to lower the loss, and the *update* moves the weights a small step, the learning rate, in that direction. `Xs @ w` multiplies the 2000 × 2 table by the two weights. `for step in range(201)` repeats the indented lines 201 times, and `step % 50 == 0` is true every 50 steps.

**Quick check.** `X12` has the shape (12, 64, 64). What is the shape of `X12[:, 10:20, 10:20]`?

<details><summary>Answer</summary>

(12, 10, 10): all 12 frames, rows 10 to 19 and columns 10 to 19.

</details>

**Exercise 3.1.** How many numbers does `X12` hold in total?

<details><summary><b>Prompt route</b> for Exercise 3.1: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `X12` is a NumPy array of shape (12, 64, 64): 12 frames of 64 × 64 pixels, values from 0 to 1.
Task: Count all the numbers that `X12` holds.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The count as an integer. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise 3.1", answer)`. Return only the code in one block, with a short comment above each step.
Test: Check the method first on `np.zeros((2, 3, 4))`, which holds 24 numbers.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise 3.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(X12.size)
teach.check("Exercise 3.1", answer)

**Exercise 3.2.** What is the mean value of the pixels of `frame`, the image read from VC-1212.png? Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise 3.2: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `frame` is a NumPy array of shape (64, 64) read with `plt.imread` from the file VC-1212.png, values from 0 to 1.
Task: Compute the mean value of all its pixels.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The mean, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise 3.2", answer)`. Return only the code in one block, with a short comment above each step.
Test: Print the minimum and the maximum as well; the mean must lie between them, and the maximum must not exceed 1.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise 3.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(frame.mean()), 3)
teach.check("Exercise 3.2", answer)

**Exercise 3.3.** Train the neuron again from zero weights, but with the learning rate 0.05 instead of 0.5. What is the loss after 200 updates? Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise 3.3: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `Xs` is a NumPy array of shape (2000, 2) with the standardised lvedd_mm and ivs_mm of the 2000 patients (mean 0, SD 1). `target` is a NumPy array of 0 or 1 (1 abnormal).
Task: Train one logistic neuron by gradient descent: start from w = [0, 0] and b = 0, and make 200 updates with the learning rate 0.05. Each update computes p = sigmoid(Xs @ w + b), then subtracts 0.05 times the gradients Xs.T @ (p − target) / n and mean(p − target). After the 200 updates, compute the mean binary cross-entropy.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The loss after the 200 updates, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise 3.3", answer)`. Return only the code in one block, with a short comment above each step.
Test: With the learning rate 0.5 of the Python step, the same loop must give the loss printed there at step 200.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise 3.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
w3, b3 = np.zeros(2), 0.0
for step in range(200):
    p3 = sigmoid(Xs @ w3 + b3)
    w3 = w3 - 0.05 * Xs.T @ (p3 - target) / len(target)
    b3 = b3 - 0.05 * np.mean(p3 - target)
p3 = sigmoid(Xs @ w3 + b3)
answer = round(float(-np.mean(target * np.log(p3) + (1 - target) * np.log(1 - p3))), 3)
teach.check("Exercise 3.3", answer)

## 1. A neuron and a network in NumPy

*Lecture: [Neural networks in one page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#neural-networks-in-one-page)*

A neuron adds its inputs, each with its own weight, adds a bias and passes the sum through an *activation function*. A layer is a row of neurons that see the same inputs, and a network stacks layers so that each layer works on the outputs of the one before. Without a non-linear activation, a stack of layers would collapse into a single straight boundary.

**From the formulas to the code.** The weighted sum $z = \mathbf{w} \cdot \mathbf{x} + b$ is `w @ x + b`. The sigmoid $\sigma(z) = 1/(1 + e^{-z})$ is `sigmoid(z)`, the rectified linear unit (ReLU) $\max(0, z)$ is `np.maximum(0, z)`, and the softmax $e^{z_k}/\sum_j e^{z_j}$ turns the outputs of the last layer into probabilities that add up to 1. A layer with weights `W1` of shape (inputs, neurons) computes `x @ W1 + b1` for all its neurons at once.

**Started here?** Run the catch-up cell below once. It runs the code that Section 1 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(1)

In [ ]:
# Three activation functions
zz = np.linspace(-5, 5, 201)
fig, ax = plt.subplots(figsize=(6.2, 3.2))
ax.plot(zz, sigmoid(zz), color="#9B1B1B", lw=2, label="sigmoid")
ax.plot(zz, np.tanh(zz), color="#1F5F8B", lw=2, label="tanh")
ax.plot(zz, np.maximum(0, zz), color="#B97813", lw=2, label="ReLU")
ax.set_ylim(-1.2, 3)
ax.axhline(0, color="#888888", lw=0.8)
ax.set_xlabel("z")
ax.legend()
plt.show()

In [ ]:
# A small network: 2 inputs, 3 hidden neurons with ReLU, 2 outputs with softmax
def softmax(v):
    e = np.exp(v - v.max())
    return e / e.sum()


W1 = np.array([[1.5, -1.0, 0.5],
               [0.2, 1.2, -0.8]])
b1 = np.array([0.0, -0.5, 0.1])
W2 = np.array([[-1.0, 1.0],
               [-1.2, 1.2],
               [0.8, -0.8]])
b2 = np.array([0.6, -0.6])

x_case = Xs[registry["patient_id"] == "VC-1212"][0]     # standardised lvedd and ivs of the case
hidden = np.maximum(0, x_case @ W1 + b1)
out = softmax(hidden @ W2 + b2)
print("x =", np.round(x_case, 3))
print("hidden layer =", np.round(hidden, 3))
print("probabilities (normal, abnormal) =", np.round(out, 3))

**Python note.** `np.array([[...], [...]])` writes a table row by row: `W1` has 2 rows (inputs) and 3 columns (neurons). `x_case @ W1` gives one weighted sum per hidden neuron. Subtracting the largest value inside `softmax` changes nothing in the result, but it keeps `np.exp` from overflowing for large numbers. The weights here are set by hand; Section 2 lets the network learn them.

**Observe and reflect.** Replace `np.maximum(0, ...)` by the identity, `x_case @ W1 + b1`, and work out on paper what the two layers become together. Why does a network need a non-linear activation between its layers?

**Exercise A.** A neuron receives the two standardised measurements `x = [1.2, -0.5]`. Its weights are `w = [0.8, -1.5]` and its bias `b = 0.1`, and it uses the sigmoid. What is its output? Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise A: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: Two standardised measurements `x = [1.2, -0.5]`, the weights `w = [0.8, -1.5]` and the bias `b = 0.1`.
Task: Compute the weighted sum z = w · x + b and the output of the sigmoid, 1 / (1 + e^(−z)).
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The output of the sigmoid, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise A", answer)`. Return only the code in one block, with a short comment above each step.
Test: With w = [0, 0] and b = 0 the output must be 0.5.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
z_a = np.array([0.8, -1.5]) @ np.array([1.2, -0.5]) + 0.1
answer = round(float(sigmoid(z_a)), 3)
teach.check("Exercise A", answer)

**Prompt challenge, Section 1: Activation functions and their slopes.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: No data: a grid of z values from −5 to 5.
Task: Draw the sigmoid, tanh and ReLU in one panel and their derivatives in a second panel.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure with two panels. Return only the code in one block, with a short comment above each step.
Test: Print the largest value of each derivative.
</pre>

</details>

**What to check.** The derivative of the sigmoid peaks at 0.25 at z = 0, that of tanh at 1; the derivative of ReLU is 0 for negative z and 1 for positive z.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 2. Learning: Loss, gradients and the training loop in PyTorch

*Lecture: [Learning: Loss, gradients and the training loop](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#learning-loss-gradients-and-the-training-loop)*

A network learns by lowering a *loss*, a number that grows when its predictions are wrong. *Backpropagation* applies the chain rule of calculus from the loss back to every weight and gives the *gradient*, the direction in which the loss rises fastest. *Gradient descent* moves each weight a small step against its gradient. PyTorch computes the gradients by itself: This is called *autograd*.

**From the formulas to the code.** For one patient with label $y$ and predicted probability $p$, the binary cross-entropy is $L = -[y \ln p + (1-y) \ln(1-p)]$, `F.binary_cross_entropy(p, y)`. For a neuron with the sigmoid, its gradient is $\partial L/\partial w_j = (p - y)\,x_j$. The update $w \leftarrow w - \eta\,\partial L/\partial w$ is what `optimizer.step()` does, after `loss.backward()` has filled `w.grad`. With several classes, `F.cross_entropy(logits, labels)` applies the softmax and the cross-entropy in one step.

**Started here?** Run the catch-up cell below once. It runs the code that Section 2 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(2)

In [ ]:
# Autograd checks the formula of the gradient
x_t = torch.tensor([1.2, -0.5])
w_t = torch.tensor([0.8, -1.5], requires_grad=True)
b_t = torch.tensor(0.1, requires_grad=True)
p_t = torch.sigmoid(w_t @ x_t + b_t)
loss_t = F.binary_cross_entropy(p_t, torch.tensor(1.0))
loss_t.backward()
print("p =", round(p_t.item(), 3), " loss =", round(loss_t.item(), 4))
print("autograd:", np.round(w_t.grad.numpy(), 4), round(b_t.grad.item(), 4))
print("formula (p − y) · x:", np.round((p_t.item() - 1.0) * x_t.numpy(), 4))

**Python note.** A tensor is PyTorch's array. `requires_grad=True` asks PyTorch to remember every operation done with the tensor, so that `loss_t.backward()` can go back through them and store the gradient in `w_t.grad`. `.item()` turns a tensor with one value into a Python number, and `.numpy()` turns a tensor into a NumPy array.

In [ ]:
# A network with 8 hidden neurons learns the report from the two measurements
Xs_t = torch.tensor(Xs, dtype=torch.float32)
t_t = torch.tensor(target)
torch.manual_seed(RANDOM_STATE)
mlp = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 2))
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.05)
for step in range(301):
    loss = F.cross_entropy(mlp(Xs_t), t_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        print(step, round(loss.item(), 4))

g1, g2 = np.meshgrid(np.linspace(-3, 5, 200), np.linspace(-3, 5, 200))
grid = np.c_[g1.ravel(), g2.ravel()]
with torch.no_grad():
    p_mlp = F.softmax(mlp(torch.tensor(grid, dtype=torch.float32)), dim=1)[:, 1].numpy().reshape(g1.shape)
p_one = sigmoid(grid @ w + b).reshape(g1.shape)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
for ax, pr, name in [(axes[0], p_one, "one neuron"), (axes[1], p_mlp, "8 hidden neurons")]:
    ax.contourf(g1, g2, pr, levels=[0, 0.5, 1], colors=["#EAF1F7", "#F6E0E0"])
    ax.scatter(Xs[target == 0, 0], Xs[target == 0, 1], s=4, color="#1F5F8B", alpha=0.4, label="normal")
    ax.scatter(Xs[target == 1, 0], Xs[target == 1, 1], s=6, color="#9B1B1B", label="abnormal")
    ax.set_title(name)
    ax.set_xlabel("lvedd_mm (z)")
axes[0].set_ylabel("ivs_mm (z)")
axes[1].legend(loc="upper right")
plt.show()
with torch.no_grad():
    acc_mlp = (mlp(Xs_t).argmax(1).numpy() == target).mean()
print("accuracy: one neuron", round(((sigmoid(Xs @ w + b) > 0.5) == target).mean(), 3),
      "| network", round(acc_mlp, 3))

**Python note.** `nn.Sequential` chains layers: `nn.Linear(2, 8)` is a layer of 8 neurons with 2 inputs each, then ReLU, then a layer of 2 output neurons. Each loop has the four steps of the Python step: the forward pass and the loss in one line, `optimizer.zero_grad()` to clear the old gradients, `loss.backward()` for the new ones and `optimizer.step()` for the update. `torch.no_grad()` turns off the bookkeeping of autograd when no learning is needed.

**Observe and reflect.** The report calls a frame abnormal when the ventricle is dilated *or* the septum is thick. Why can one neuron draw only one straight boundary in the left panel, while the network with a hidden layer bends its boundary around the corner?

**Exercise B.** One patient has `x = [1.0, 2.0]` and the label `y = 1`. A neuron with the sigmoid has the weights `w = [0.5, -0.25]` and the bias 0. Compute the binary cross-entropy with PyTorch, call `backward()` and give the gradient of the loss with respect to the first weight. Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise B: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: One patient with `x = [1.0, 2.0]` and the label `y = 1`, and one neuron with the weights `w = [0.5, -0.25]`, the bias 0 and the sigmoid. The loss is the binary cross-entropy.
Task: Make w a tensor with `requires_grad=True`, compute the loss with `F.binary_cross_entropy`, call `backward()` and read the gradient of the loss with respect to the first weight.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The gradient with respect to w1, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise B", answer)`. Return only the code in one block, with a short comment above each step.
Test: Compare it with the formula (p − y) · x1; the two values must be equal.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
w_b = torch.tensor([0.5, -0.25], requires_grad=True)
p_b = torch.sigmoid(w_b @ torch.tensor([1.0, 2.0]))
F.binary_cross_entropy(p_b, torch.tensor(1.0)).backward()
answer = round(float(w_b.grad[0]), 3)
teach.check("Exercise B", answer)

**Prompt challenge, Section 2: Three learning rates.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `Xs` is a NumPy array of shape (2000, 2) with the standardised lvedd_mm and ivs_mm of the 2000 patients (mean 0, SD 1). `target` is a NumPy array of 0 or 1 (1 abnormal).
Task: Train the logistic neuron of the Python step for 300 updates with the learning rates 0.01, 0.1 and 1.0, starting each time from zero weights, and plot the loss against the update for the three rates.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure with three lines and the final loss of each rate. Return only the code in one block, with a short comment above each step.
Test: The losses must start at the same value, log(2) = 0.693.
</pre>

</details>

**What to check.** The rate 0.01 ends well above the other two; 1.0 falls fastest. Too small a rate wastes steps.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 3. The frames of the echo laboratory

*Lecture: [Images as arrays: The frames of the echo laboratory](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#images-as-arrays-the-frames-of-the-echo-laboratory)*

The echo laboratory of the virtual clinic has one end-diastolic frame of the parasternal short-axis view for each of the 2000 patients. The size of the ventricle and the thickness of its walls in each frame come from the patient's measurements in the registry. The report of the laboratory calls a frame abnormal when the left ventricle is dilated (diameter above 58 mm) or the septum is thick (13 mm or more). About one frame in ten is abnormal.

**From the formulas to the code.** The frames are split into a training set (60 percent), a validation set (20 percent) for choosing the epoch, and a test set (20 percent) that is used once at the end. Each frame is normalised as $x' = (x - \mu)/\sigma$ with the mean $\mu$ and the SD $\sigma$ of the *training* frames. PyTorch expects images with the shape (number, channels, rows, columns), here (n, 1, 64, 64).

**Started here?** Run the catch-up cell below once. It runs the code that Section 3 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(3)

In [ ]:
# All 2000 frames of the echo laboratory
X, y, echo_table = echo_images(registry)
print(X.shape, "abnormal:", int(y.sum()), "/", len(y))
print(echo_table[["dilated", "thick_septum"]].sum().to_dict())
show = list(np.where(y == 0)[0][:4]) + list(np.where(y == 1)[0][:4])
fig, axes = plt.subplots(2, 4, figsize=(9, 4.9))
for ax, i in zip(axes.ravel(), show):
    ax.imshow(X[i], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{echo_table.patient_id[i]} ({echo_table.report[i]})\n{echo_table.lvedd_mm[i]:.0f} / {echo_table.ivs_mm[i]:.1f} mm", fontsize=8)
    ax.axis("off")
plt.show()

In [ ]:
# Training, validation and test sets, normalised with the training frames
idx = np.arange(len(y))
train_idx, test_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
train_idx, val_idx = train_test_split(train_idx, test_size=0.25, stratify=y[train_idx], random_state=RANDOM_STATE)
mean, sd = float(X[train_idx].mean()), float(X[train_idx].std())


def to_tensor(rows):
    return torch.tensor((X[rows] - mean) / sd, dtype=torch.float32).unsqueeze(1)


Xtr, Xva, Xte = to_tensor(train_idx), to_tensor(val_idx), to_tensor(test_idx)
ytr, yva, yte = (torch.tensor(y[rows]) for rows in (train_idx, val_idx, test_idx))
print(Xtr.shape, Xva.shape, Xte.shape, "| mean", round(mean, 4), " SD", round(sd, 4))
print("abnormal per set:", int(ytr.sum()), int(yva.sum()), int(yte.sum()))

**Python note.** `train_test_split` is called twice: first it keeps 20 percent for the test, then 25 percent of the rest, which is 20 percent of all, for the validation. `stratify=y` keeps the share of abnormal frames the same in every set. `.unsqueeze(1)` adds the channel dimension at position 1, so (n, 64, 64) becomes (n, 1, 64, 64). The line with three names on the left unpacks three tensors made by a *generator expression*, a loop written inside parentheses.

**Exercise C.** How many abnormal frames does the test set hold?

<details><summary><b>Prompt route</b> for Exercise C: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `X` is a NumPy array of shape (2000, 64, 64) with one end-diastolic parasternal short-axis frame per patient of the virtual clinic, pixel values from 0 to 1, one pixel = 2 mm. `y` holds the report of the echo laboratory, 0 or 1 (1 abnormal: a left ventricular end-diastolic diameter above 58 mm or a septum of 13 mm or more). `echo_table` holds patient_id, lvedd_mm, ivs_mm and pw_mm (mm), and dilated and thick_septum (0 or 1). `train_idx`, `val_idx` and `test_idx` are the indices of 1200, 400 and 400 frames, split with stratification by `y`. `Xtr`, `Xva` and `Xte` are float tensors of shape (n, 1, 64, 64), normalised with the mean and the SD of the training frames; `ytr`, `yva` and `yte` are tensors of labels, 0 or 1.
Task: Count the abnormal frames (label 1) of the test set.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The count as an integer. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise C", answer)`. Return only the code in one block, with a short comment above each step.
Test: The training, validation and test sets must add up to 2000 frames, and the share of abnormal frames must be about the same in the three sets.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(yte.sum())
teach.check("Exercise C", answer)

**Prompt challenge, Section 3: Normal and abnormal frames side by side.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `X` is a NumPy array of shape (2000, 64, 64) with one end-diastolic parasternal short-axis frame per patient of the virtual clinic, pixel values from 0 to 1, one pixel = 2 mm. `y` holds the report of the echo laboratory, 0 or 1 (1 abnormal: a left ventricular end-diastolic diameter above 58 mm or a septum of 13 mm or more). `echo_table` holds patient_id, lvedd_mm, ivs_mm and pw_mm (mm), and dilated and thick_septum (0 or 1).
Task: Show four normal and four abnormal frames in a grid of 2 × 4, each with its lvedd_mm and ivs_mm in the title.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure. Return only the code in one block, with a short comment above each step.
Test: Print the label of each frame shown; the first row must be all 0 and the second all 1.
</pre>

</details>

**What to check.** The abnormal frames show either a large cavity or a thick wall; some of the thick walls are hard to tell from normal ones by eye.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 4. Convolution by hand and with PyTorch

*Lecture: [Convolution: Filters that slide over an image](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#convolution-filters-that-slide-over-an-image)*

A convolution slides a small table of weights, the *kernel*, over the image. At each position it multiplies the kernel with the pixels under it and adds the products, which gives one pixel of the output, the *feature map*. The same kernel is used everywhere, so a layer needs few weights, and a pattern is found wherever it appears. *Max pooling* keeps the largest value of each 2 × 2 block and halves the size.

**From the formulas to the code.** The output is $Y(i, j) = \sum_{u}\sum_{v} K(u, v)\,X(i+u, j+v) + b$, `F.conv2d(x, kernel, bias)`. With an input of $n$ pixels, a kernel of $k$, padding $p$ and stride $s$, the output has $\lfloor (n + 2p - k)/s \rfloor + 1$ pixels, `conv_out(n, k, s, p)`. Strictly, this formula is a cross-correlation; deep learning calls it convolution because the network learns the kernel either way.

**Started here?** Run the catch-up cell below once. It runs the code that Section 4 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(4)

In [ ]:
# A 3 × 3 kernel on a 5 × 5 patch, by hand and with PyTorch
patch = np.array([[0, 0, 0, 0, 0],
                  [0, 1, 1, 1, 0],
                  [0, 1, 0, 1, 0],
                  [0, 1, 1, 1, 0],
                  [0, 0, 0, 0, 0]], dtype=float)
kernel = np.array([[-1, 0, 1],
                   [-1, 0, 1],
                   [-1, 0, 1]], dtype=float)          # vertical edges
by_hand = np.zeros((3, 3))
for r in range(3):
    for c in range(3):
        by_hand[r, c] = np.sum(patch[r:r + 3, c:c + 3] * kernel)
print(by_hand)
with_torch = F.conv2d(torch.tensor(patch)[None, None], torch.tensor(kernel)[None, None])
print(with_torch[0, 0].numpy())


def conv_out(n, k, s=1, p=0):
    return (n + 2 * p - k) // s + 1


for n, k, s, p in [(64, 3, 1, 1), (64, 3, 1, 0), (64, 5, 2, 2), (32, 2, 2, 0)]:
    print(f"n={n} k={k} s={s} p={p} -> {conv_out(n, k, s, p)}")

**Python note.** Two nested `for` loops visit every position of the 3 × 3 output, and `patch[r:r + 3, c:c + 3]` is the 3 × 3 window at that position. `[None, None]` adds two dimensions in front, because `F.conv2d` expects (number, channels, rows, columns) for the image and (output channels, input channels, rows, columns) for the kernel. `//` divides and drops the remainder.

In [ ]:
# Three kernels on the frame of VC-1873, a dilated ventricle
frame_d = plt.imread(ROOT / "data" / "examples" / "echo" / "VC-1873.png")
kernels = {"vertical edges": [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
           "horizontal edges": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
           "blur": [[1 / 9, 1 / 9, 1 / 9]] * 3}
t = torch.tensor(frame_d)[None, None]
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(frame_d, cmap="gray")
axes[0].set_title("VC-1873")
for ax, (name, k) in zip(axes[1:], kernels.items()):
    fmap = F.conv2d(t, torch.tensor(k, dtype=torch.float32)[None, None], padding=1)[0, 0]
    ax.imshow(fmap.numpy(), cmap="RdBu_r")
    ax.set_title(name)
for ax in axes:
    ax.axis("off")
plt.show()
print(tuple(t.shape), "→ max pooling →", tuple(F.max_pool2d(t, 2).shape))

**Python note.** A dictionary keeps names and values in pairs, and `.items()` returns the pairs one by one for the loop. `padding=1` adds a frame of zeros around the image, so a 3 × 3 kernel keeps the size at 64 × 64. The colour map `RdBu_r` shows negative values in blue and positive values in red, which is how an edge filter answers on the two sides of a wall.

**Exercise D.** A convolution with a 5 × 5 kernel, stride 2 and padding 1 runs over an input of 64 × 64 pixels. What is the size of one side of the output?

<details><summary><b>Prompt route</b> for Exercise D: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: A square input of 64 × 64 pixels with one channel.
Task: Compute the output size of a convolution with a 5 × 5 kernel, stride 2 and padding 1 with the formula floor((n + 2p − k) / s) + 1, and check it with `F.conv2d` on `torch.zeros(1, 1, 64, 64)`.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The output size (one side) as an integer. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise D", answer)`. Return only the code in one block, with a short comment above each step.
Test: With a 3 × 3 kernel, stride 1 and padding 1, the size must stay 64.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = conv_out(64, 5, 2, 1)
print(F.conv2d(torch.zeros(1, 1, 64, 64), torch.zeros(1, 1, 5, 5), stride=2, padding=1).shape)
teach.check("Exercise D", answer)

**Prompt challenge, Section 4: Two filters on a dilated ventricle.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `frame_d` is the frame of VC-1873, a NumPy array of shape (64, 64), values from 0 to 1.
Task: Apply a 3 × 3 averaging kernel (all values 1/9) and a 3 × 3 Laplacian kernel [[0, 1, 0], [1, −4, 1], [0, 1, 0]] with `F.conv2d` and padding 1, and show the frame and the two results side by side.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure with three panels. Return only the code in one block, with a short comment above each step.
Test: Both results must have the shape (64, 64).
</pre>

</details>

**What to check.** The averaging kernel smooths the speckle; the Laplacian lights up the borders of the myocardium.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 5. A first convolutional network

*Lecture: [A convolutional network for the echo laboratory](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#a-convolutional-network-for-the-echo-laboratory)*

The network of the echo laboratory has three blocks. Each block has a convolution with 3 × 3 kernels, batch normalisation, ReLU and max pooling. The frame shrinks from 64 to 32, 16 and 8 pixels, while the number of feature maps grows from 8 to 16 and 32. A linear layer turns the 32 × 8 × 8 = 2048 values into two outputs, one for each class. The training loop of Section 2 now works on *mini-batches* of 32 frames, and one pass over all training frames is an *epoch*. After each epoch, the loss on the validation frames decides which weights to keep.

**From the formulas to the code.** The class `EchoCNN` lists the layers in `__init__` and says in `forward` how a frame passes through them. `train_model` runs the epochs and returns the training and validation loss of each epoch. `F.softmax(model(x), dim=1)[:, 1]` is the probability of the abnormal class.

**Started here?** Run the catch-up cell below once. It runs the code that Section 5 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(5)

In [ ]:
# The network of the echo laboratory
class EchoCNN(nn.Module):
    """Three convolution blocks and a linear layer for frames of 64 × 64 pixels."""

    def __init__(self, n_classes=2, dropout=0.3, batchnorm=True):
        super().__init__()
        layers, c_in = [], 1
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


print(EchoCNN())
with torch.no_grad():
    print(EchoCNN()(Xtr[:4]).shape)

**Python note.** A *class* is a recipe for objects. `class EchoCNN(nn.Module)` builds on PyTorch's `nn.Module`, and `super().__init__()` lets the parent prepare itself first. `self.features` stores the layers inside the object. `layers += [...]` appends to a list, and `[...] if batchnorm else []` adds batch normalisation only when it is asked for. `nn.Sequential(*layers)` unpacks the list into separate arguments.

**Exercise E.** How many trainable parameters does `EchoCNN(n_classes=2)` have?

<details><summary><b>Prompt route</b> for Exercise E: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `EchoCNN` is the class of Section 5: three blocks of Conv2d (3 × 3, padding 1), BatchNorm2d, ReLU and MaxPool2d(2) with 8, 16 and 32 channels, then Flatten, Dropout(0.3) and Linear(2048, n_classes).
Task: Create `EchoCNN(n_classes=2)` and count its trainable parameters.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The number of parameters as an integer. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise E", answer)`. Return only the code in one block, with a short comment above each step.
Test: Count the parameters of `nn.Conv2d(1, 8, 3)` as well; it must have 80.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = sum(p.numel() for p in EchoCNN(n_classes=2).parameters() if p.requires_grad)
teach.check("Exercise E", answer)

In [ ]:
# Training with mini-batches; the weights of the best epoch are kept
def train_model(model, Xtr, ytr, Xva, yva, epochs=15, lr=3e-3, batch=32, augment_fn=None, seed=RANDOM_STATE):
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history, best_loss, best_state = [], float("inf"), None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(ytr))
        total = 0.0
        for start in range(0, len(ytr), batch):
            j = order[start:start + batch]
            xb = augment_fn(Xtr[j]) if augment_fn else Xtr[j]
            loss = F.cross_entropy(model(xb), ytr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(j)
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(Xva), yva).item()
        history.append((total / len(ytr), val_loss))
        if val_loss < best_loss:
            best_loss, best_state = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return history

**Python note.** `model.train()` and `model.eval()` switch dropout and batch normalisation between their training and their evaluation behaviour. `torch.randperm` shuffles the order of the frames in every epoch. `copy.deepcopy(model.state_dict())` saves a full copy of the weights, and `load_state_dict` puts the best copy back at the end. The loss of each batch is weighted by its size, `len(j)`, so the last, smaller batch counts correctly.

In [ ]:
# Train the network for 15 epochs and draw the two losses
torch.manual_seed(RANDOM_STATE)
model = EchoCNN()
history = train_model(model, Xtr, ytr, Xva, yva)
best_epoch = int(np.argmin([v for t, v in history])) + 1
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(range(1, len(history) + 1), [t for t, v in history], color="#1F5F8B", marker="o", ms=3, label="training")
ax.plot(range(1, len(history) + 1), [v for t, v in history], color="#9B1B1B", marker="o", ms=3, label="validation")
ax.axvline(best_epoch, color="#888888", ls="--", lw=1)
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.legend()
plt.show()
print("best epoch:", best_epoch)

In [ ]:
# The network on the 400 test frames
model.eval()
with torch.no_grad():
    p_test = F.softmax(model(Xte), dim=1)[:, 1].numpy()
y_test = yte.numpy()
pred = (p_test >= 0.5).astype(int)
print(confusion_matrix(y_test, pred))
print("accuracy", round(accuracy_score(y_test, pred), 3),
      "| precision", round(precision_score(y_test, pred), 3),
      "| recall", round(recall_score(y_test, pred), 3),
      "| F1", round(f1_score(y_test, pred), 3),
      "| AUC", round(roc_auc_score(y_test, p_test), 3))

**Observe and reflect.** The course trained the same network with the same seed and reached an accuracy of 0.968, an F1 of 0.822 and an area under the receiver operating characteristic (ROC) curve (AUC) of 0.983. Your numbers may differ in the last digits, because computers add the numbers of a convolution in different orders. Which of the measures would you report to the head of the echo laboratory, and why not the accuracy alone?

**Prompt challenge, Section 5: Accuracy per epoch.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `train_idx`, `val_idx` and `test_idx` are the indices of 1200, 400 and 400 frames, split with stratification by `y`. `Xtr`, `Xva` and `Xte` are float tensors of shape (n, 1, 64, 64), normalised with the mean and the SD of the training frames; `ytr`, `yva` and `yte` are tensors of labels, 0 or 1. `EchoCNN` is the class of Section 5: three blocks of Conv2d (3 × 3, padding 1), BatchNorm2d, ReLU and MaxPool2d(2) with 8, 16 and 32 channels, then Flatten, Dropout(0.3) and Linear(2048, n_classes).
Task: Train a new `EchoCNN` for 10 epochs with Adam (learning rate 0.003, batches of 32) and record the accuracy on the training set and on the validation set after every epoch. Plot both against the epoch.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure with two lines. Return only the code in one block, with a short comment above each step.
Test: Compute the accuracy in evaluation mode and inside `torch.no_grad()`; both values must lie between 0 and 1.
</pre>

</details>

**What to check.** Both curves pass 0.9 after the first epochs, because 90 percent of the frames are normal. The F1 of Section 7 tells more.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 6. Overfitting: Dropout, batch normalisation and augmentation

*Lecture: [Overfitting: Dropout, batch normalisation and augmentation](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#overfitting-dropout-batch-normalisation-and-augmentation)*

With few training frames, a network can learn them by heart: The training loss falls toward zero while the validation loss stops falling or rises. Three tools work against this. *Dropout* switches off a random share $p$ of the units in each training step. *Batch normalisation* rescales each feature map with the mean and the SD of the mini-batch. *Augmentation* shows the network slightly changed copies of the frames, here small rotations, shifts and changes of brightness. *Early stopping* keeps the weights of the epoch with the lowest validation loss, as `train_model` already does.

**From the formulas to the code.** In training, dropout multiplies the kept units by $1/(1-p)$, so their expected sum stays the same; in evaluation it does nothing. Batch normalisation computes $\hat{x} = (x - \mu_B)/\sqrt{\sigma_B^2 + \varepsilon}$ and then $\gamma \hat{x} + \beta$, with $\gamma$ and $\beta$ learned. `augment(x)` builds a rotation and a shift for every frame with `F.affine_grid` and applies them with `F.grid_sample`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 6 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(6)

In [ ]:
# Augmentation: small rotations, shifts and changes of brightness
def augment(x, max_deg=15, max_shift=0.08, gain=0.10):
    n = x.shape[0]
    a = (torch.rand(n) * 2 - 1) * math.radians(max_deg)
    s = (torch.rand(n, 2) * 2 - 1) * max_shift
    theta = torch.stack([torch.stack([a.cos(), -a.sin(), s[:, 0]], 1),
                         torch.stack([a.sin(), a.cos(), s[:, 1]], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, padding_mode="border", align_corners=False)
    return x * (1 + (torch.rand(n, 1, 1, 1) * 2 - 1) * gain)


torch.manual_seed(RANDOM_STATE)
case_row = list(echo_table.patient_id).index("VC-1212")
copies = augment(to_tensor([case_row]).repeat(4, 1, 1, 1))
fig, axes = plt.subplots(1, 5, figsize=(11, 2.5))
axes[0].imshow(X[case_row], cmap="gray")
axes[0].set_title("VC-1212")
for k, ax in enumerate(axes[1:]):
    ax.imshow(copies[k, 0].numpy(), cmap="gray")
    ax.set_title(f"copy {k + 1}")
for ax in axes:
    ax.axis("off")
plt.show()

**Python note.** `torch.rand(n)` draws n numbers between 0 and 1, so `torch.rand(n) * 2 - 1` lies between −1 and 1. `theta` holds one 2 × 3 matrix per frame: The first two columns rotate, the last one shifts. `.repeat(4, 1, 1, 1)` makes four copies of the frame along the first dimension. The augmentation does not flip the frame from left to right: That would put the right ventricle on the wrong side of the heart.

In [ ]:
# 300 training frames, 40 epochs: without and with augmentation
torch.manual_seed(RANDOM_STATE)
plain = EchoCNN(dropout=0.0, batchnorm=False)
hist_plain = train_model(plain, Xtr[:300], ytr[:300], Xva, yva, epochs=40)
torch.manual_seed(RANDOM_STATE)
augmented = EchoCNN(dropout=0.0, batchnorm=False)
hist_aug = train_model(augmented, Xtr[:300], ytr[:300], Xva, yva, epochs=40, augment_fn=augment)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, h, name in [(axes[0], hist_plain, "no regularisation"), (axes[1], hist_aug, "augmentation")]:
    ax.plot(range(1, 41), [t for t, v in h], color="#1F5F8B", label="training")
    ax.plot(range(1, 41), [v for t, v in h], color="#9B1B1B", label="validation")
    ax.set_title(name)
    ax.set_xlabel("epoch")
    ax.set_yscale("log")
axes[0].set_ylabel("loss (log scale)")
axes[1].legend()
plt.show()
for name, h in [("plain", hist_plain), ("augmented", hist_aug)]:
    print(name, "last epoch: training", round(h[-1][0], 4), "validation", round(h[-1][1], 4))

**Observe and reflect.** Without regularisation, the training loss falls far below the validation loss: The network has learned the 300 frames by heart. With augmentation the two curves stay close. Why does a rotated copy of a frame count as new information for the network, but not for a cardiologist?

In [ ]:
# Dropout and batch normalisation on small tensors
torch.manual_seed(RANDOM_STATE)
ones = torch.ones(10)
print("dropout, training:  ", F.dropout(ones, p=0.3, training=True).numpy().round(4))
print("dropout, evaluation:", F.dropout(ones, p=0.3, training=False).numpy())
bn = nn.BatchNorm1d(1)
batch_of_four = torch.tensor([[2.0], [4.0], [6.0], [8.0]])
print("batch normalisation of 2, 4, 6, 8:", bn(batch_of_four).detach().flatten().numpy().round(3))

**Exercise F.** In training mode, dropout with p = 0.3 sets 30 percent of the values to 0 on average. What value do the kept elements of `torch.ones(10)` take? Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise F: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: A tensor of ten ones, `torch.ones(10)`.
Task: Apply `F.dropout` with p = 0.3 in training mode and find the value that the kept elements take.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The value of a kept element, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise F", answer)`. Return only the code in one block, with a short comment above each step.
Test: With `training=False` the tensor must come back unchanged.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
kept = F.dropout(torch.ones(10), p=0.3, training=True)
answer = round(float(kept[kept > 0][0]), 3)
teach.check("Exercise F", answer)

**Prompt challenge, Section 6: Eight augmented copies of the case.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `frame` is the frame of VC-1212, a NumPy array of shape (64, 64), and `augment(x)` is the function of Section 6, which takes a tensor of shape (n, 1, 64, 64).
Task: Make eight augmented copies of the frame with `augment` and show them in a grid of 2 × 4.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure. Return only the code in one block, with a short comment above each step.
Test: Each copy must keep the shape (64, 64) and values near the range of the frame.
</pre>

</details>

**What to check.** The heart turns and shifts by a few pixels and the brightness changes, while the anatomy stays plausible. A left-right flip would move the right ventricle to the wrong side.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 7. Reading the result: Precision, recall and F1

*Lecture: [Reading the result: Precision, recall and F1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#reading-the-result-precision-recall-and-f1)*

When nine frames in ten are normal, a network that calls every frame normal reaches an accuracy of 0.90 and finds no disease. The measures of a class answer better: The *precision* is the share of correct alarms among all alarms, the *recall* (sensitivity) the share of diseased frames that are found, and the *F1 score* their harmonic mean. With several classes, the *macro F1* is the plain mean of the F1 of the classes, so a small class counts as much as a large one. The second cell reads the result of a network on 180 short-axis slices of cine cardiac magnetic resonance (CMR) with three classes: normal, dilated cardiomyopathy (DCM) and hypertrophic cardiomyopathy (HCM).

**From the formulas to the code.** For class $k$, $P_k = TP_k/(TP_k + FP_k)$, $R_k = TP_k/(TP_k + FN_k)$ and $F1_k = 2 P_k R_k/(P_k + R_k)$. In a confusion matrix `cm` with the true class in the rows, `np.diag(cm)` holds the $TP_k$, the column sums `cm.sum(axis=0)` are $TP_k + FP_k$ and the row sums `cm.sum(axis=1)` are $TP_k + FN_k$.

**Started here?** Run the catch-up cell below once. It runs the code that Section 7 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(7)

In [ ]:
# Which abnormal frames does the network miss?
test_table = echo_table.iloc[test_idx].reset_index(drop=True)
test_table["p_abnormal"] = p_test.round(3)
test_table["pred"] = pred
dilated = test_table["dilated"] == 1
thick_only = (test_table["thick_septum"] == 1) & ~dilated
print("recall for a dilated ventricle:", round(test_table.loc[dilated, "pred"].mean(), 3), f"(n={dilated.sum()})")
print("recall for a thick septum alone:", round(test_table.loc[thick_only, "pred"].mean(), 3), f"(n={thick_only.sum()})")
test_table.loc[(test_table["label"] == 1) & (test_table["pred"] == 0),
               ["patient_id", "lvedd_mm", "ivs_mm", "dilated", "thick_septum", "p_abnormal"]]

**Python note.** `echo_table.iloc[test_idx]` picks the rows of the test frames by position, and `.reset_index(drop=True)` numbers them again from 0, in the same order as `p_test`. `~` turns True into False and back, and `&` combines two conditions. `.loc[rows, columns]` selects rows by a condition and columns by name.

In [ ]:
# Three classes by hand: a network on 180 CMR slices
cm3 = np.array([[51, 8, 1],
                [0, 60, 0],
                [6, 0, 54]])
tp = np.diag(cm3)
precision = tp / cm3.sum(axis=0)
recall = tp / cm3.sum(axis=1)
f1 = 2 * precision * recall / (precision + recall)
pd.DataFrame({"precision": precision, "recall": recall, "F1": f1}, index=CMR_CLASSES).round(3)

**Exercise G.** What is the macro F1 of the three classes of `cm3`? Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise G: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: The confusion matrix of a network on 180 short-axis CMR slices, rows true and columns predicted, in the order normal, DCM and HCM: [[51, 8, 1], [0, 60, 0], [6, 0, 54]].
Task: From the matrix, compute the precision, the recall and the F1 of each class, then their mean, the macro F1.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The macro F1, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise G", answer)`. Return only the code in one block, with a short comment above each step.
Test: The F1 of DCM must be 0.938.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(f1.mean()), 3)
teach.check("Exercise G", answer)

**Observe and reflect.** The network finds most dilated ventricles but misses most frames whose only finding is a septum of 13 to 14 mm. At 2 mm per pixel, the difference from a normal septum of 10 mm is one or two pixels, and the training set holds only about thirty such frames. What would you change first: the resolution, the number of examples or the threshold?

**Prompt challenge, Section 7: Precision and recall across thresholds.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `p_test`, the probabilities of the abnormal class for the 400 test frames, and `y_test`, their labels (0 or 1).
Task: For the thresholds 0.1, 0.2, …, 0.9, compute the precision and the recall of the abnormal class, and plot both against the threshold.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure and a table of the nine thresholds. Return only the code in one block, with a short comment above each step.
Test: At the threshold 0.5 the table must show the values of the evaluation above it.
</pre>

</details>

**What to check.** Recall falls and precision rises as the threshold grows; a lower threshold finds more thick septa at the price of more false alarms.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 8. Known architectures and skip connections

*Lecture: [Known architectures: From LeNet to ResNet](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#known-architectures-from-lenet-to-resnet)*

Convolutional networks grew from LeNet-5, which read handwritten digits with about sixty thousand weights, to networks with millions of weights and more than a hundred layers. Very deep networks became trainable with the *skip connection* of ResNet: A block adds its input to its output, $y = F(x) + x$. The gradient can then flow past the block, and a block that has nothing to add can learn $F(x) = 0$.

**From the formulas to the code.** A convolution layer with $c_{in}$ input channels, $c_{out}$ output channels and a $k \times k$ kernel has $c_{in} \cdot c_{out} \cdot k^2 + c_{out}$ parameters, the last term for the biases. A batch normalisation layer has $2c$ parameters, $\gamma$ and $\beta$ for each of its $c$ channels. `count(model)` adds the sizes of all trainable tensors.

**Started here?** Run the catch-up cell below once. It runs the code that Section 8 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(8)

In [ ]:
# Parameters of LeNet-5, of a VGG block and of a residual block
def count(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


lenet5 = nn.Sequential(nn.Conv2d(1, 6, 5), nn.Tanh(), nn.AvgPool2d(2),
                       nn.Conv2d(6, 16, 5), nn.Tanh(), nn.AvgPool2d(2), nn.Flatten(),
                       nn.Linear(16 * 5 * 5, 120), nn.Tanh(), nn.Linear(120, 84), nn.Tanh(), nn.Linear(84, 10))
vgg_block = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
                          nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))


class BasicBlock(nn.Module):
    """The basic block of ResNet: two 3 × 3 convolutions and a skip connection."""

    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + x)            # the skip connection


print("LeNet-5 (32 × 32):", count(lenet5), "|", tuple(lenet5(torch.zeros(1, 1, 32, 32)).shape))
print("VGG block 64 → 128:", count(vgg_block))
print("EchoCNN:", count(EchoCNN()))
try:
    import torchvision
    print("ResNet-18 (torchvision):", count(torchvision.models.resnet18(weights=None)))
except ImportError:
    print("torchvision is not installed; ResNet-18 has 11,689,512 parameters.")

**Python note.** `bias=False` leaves out the biases of the convolutions, because the batch normalisation right after them has its own shift $\beta$. In `forward`, `out + x` is the skip connection; it needs `out` and `x` to have the same shape, which the padding of 1 keeps. `weights=None` builds ResNet-18 with random weights, so nothing is downloaded.

**Exercise H.** How many parameters does a convolution layer with 16 input channels, 32 output channels, a 3 × 3 kernel and biases have?

<details><summary><b>Prompt route</b> for Exercise H: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: A convolution layer with 16 input channels, 32 output channels, a 3 × 3 kernel and a bias for each output channel.
Task: Count its parameters with the formula c_in × c_out × k × k + c_out and check the result with `nn.Conv2d`.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The number of parameters as an integer. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise H", answer)`. Return only the code in one block, with a short comment above each step.
Test: `nn.Conv2d(1, 8, 3)` must have 80 parameters.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = count(nn.Conv2d(16, 32, 3))
teach.check("Exercise H", answer)

**Exercise I.** How many trainable parameters does `BasicBlock(64)` have?

<details><summary><b>Prompt route</b> for Exercise I: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: A basic block of ResNet with 64 channels: two 3 × 3 convolutions with padding 1 and without bias, each followed by `nn.BatchNorm2d(64)`, ReLU after the first, and the input added to the output before the last ReLU.
Task: Write the block as a subclass of `nn.Module` and count its trainable parameters.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The number of parameters as an integer. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise I", answer)`. Return only the code in one block, with a short comment above each step.
Test: The output of the block for `torch.randn(1, 64, 16, 16)` must have the same shape as its input.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = count(BasicBlock(64))
print(BasicBlock(64)(torch.randn(1, 64, 16, 16)).shape)
teach.check("Exercise I", answer)

**Prompt challenge, Section 8: Parameters layer by layer.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `EchoCNN` is the class of Section 5: three blocks of Conv2d (3 × 3, padding 1), BatchNorm2d, ReLU and MaxPool2d(2) with 8, 16 and 32 channels, then Flatten, Dropout(0.3) and Linear(2048, n_classes).
Task: Build a table with one row for each layer of `EchoCNN(n_classes=2)` that has parameters: its name, its type and its number of parameters.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: A pandas DataFrame. Return only the code in one block, with a short comment above each step.
Test: The column of the parameters must add up to the total of Exercise E.
</pre>

</details>

**What to check.** The linear layer holds 4098 of the 10098 parameters; the three convolutions together hold 5888.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 9. AI pair programming, part 3: Reviewing a training loop

*Lecture: [AI pair programming, part 3: Reviewing a training loop](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lecture.html#ai-pair-programming-part-3-reviewing-a-training-loop)*

A participant asked an AI assistant to *evaluate the network of the course on the test frames*. The code below is what came back. It runs and reports a high accuracy. Read it with the five questions of the training loop review in the lecture before you trust the number: Where do the statistics of the normalisation come from? Is the model in evaluation mode, and are the gradients off? Does the output layer fit the loss? Which measure suits a rare class?

**Started here?** Run the catch-up cell below once. It runs the code that Section 9 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(9)

In [ ]:
# Code from an AI assistant: evaluate the network of the course
course_model = EchoCNN()
course_model.load_state_dict(torch.load(ROOT / "models" / "week03" / "echo_cnn.pt", weights_only=True))
X_all = torch.tensor((X - X.mean()) / X.std(), dtype=torch.float32).unsqueeze(1)
probs = F.softmax(course_model(X_all[test_idx]), dim=1)
accuracy = (probs.argmax(dim=1).numpy() == y[test_idx]).mean()
print("Accuracy:", round(float(accuracy), 3))

**Observe and reflect.** The code has three problems and one omission. The statistics of the normalisation come from all 2000 frames. The model is left in training mode, so dropout switches units off at random and batch normalisation uses the statistics of the test frames. Autograd also keeps track of every operation. Accuracy alone hides that one abnormal frame in four is missed. Run the cell twice: Why does the accuracy change?

**Exercise J.** Evaluate `course_model` correctly: Switch it to evaluation mode, predict the class of every test frame of `Xte` inside `torch.no_grad()` and compute the F1 of the abnormal class against `yte`. Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise J: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-03/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `course_model` is an `EchoCNN` with the weights of the course, loaded from models/week03/echo_cnn.pt. `Xte` and `yte` are the 400 test frames and their labels (0 or 1, 40 abnormal).
Task: Switch the model to evaluation mode, predict the class with the largest output for every test frame inside `torch.no_grad()`, and compute the F1 of the abnormal class with `f1_score`.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The F1, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise J", answer)`. Return only the code in one block, with a short comment above each step.
Test: Print the accuracy as well: It must be above 0.95, while the F1 is clearly lower.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
course_model.eval()
with torch.no_grad():
    pred_course = course_model(Xte).argmax(dim=1).numpy()
print("accuracy", round(accuracy_score(yte.numpy(), pred_course), 3))
answer = round(f1_score(yte.numpy(), pred_course), 3)
teach.check("Exercise J", answer)

In [ ]:
# The case of the week, VC-1212, through the network of the course
course_model.eval()
with torch.no_grad():
    p_case = F.softmax(course_model(to_tensor([case_row])), dim=1)[0, 1].item()
print("VC-1212: probability of an abnormal frame", round(p_case, 3),
      "| lvedd", echo_table.lvedd_mm[case_row], "mm | ivs", echo_table.ivs_mm[case_row], "mm")

The [prompt workshop](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lab.html#part-f) of the interactive lab writes the prompt of such an evaluation with the seven fields of the prompt card. The field *Test* holds the check that catches the review's problems: The same evaluation run twice must give the same numbers.

**Prompt challenge, Section 9: A function that evaluates a model.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, PyTorch as torch with torch.nn as nn and torch.nn.functional as F, and scikit-learn is installed. The course package vclinic with its modules teach and imaging is available, and RANDOM_STATE = 2026.
Data: `course_model`, the `EchoCNN` of the course, and the test tensors `Xte` and `yte`.
Task: Write a function `evaluate(model, X, y)` that switches the model to evaluation mode, predicts inside `torch.no_grad()` and returns a dictionary with the accuracy, the precision, the recall and the F1 of the abnormal class.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The function and its result for the course model, rounded to three decimals. Return only the code in one block, with a short comment above each step.
Test: Calling the function twice must give the same numbers.
</pre>

</details>

**What to check.** For the course model: accuracy 0.968, precision 0.909, recall 0.75 and F1 0.822.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 10. Application key: Another data set, other settings

Change `DATASET`, `EPOCHS`, `AUGMENT` and `DROPOUT` and run the cell again. With `DATASET = "cmr"` it trains the network on the short-axis slices of the cardiomyopathy clinic, with three classes: normal, DCM and HCM. It prints the confusion matrix of the test set, the F1 of each class and the macro F1.

**Started here?** Run the catch-up cell below once. It runs the code that Section 10 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB03_deep_learning_and_cnns.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(10)

In [ ]:
DATASET = "cmr"        # "echo" or "cmr"
EPOCHS = 15
AUGMENT = True
DROPOUT = 0.3

if DATASET == "cmr":
    Xd, yd, table_d = make_cmr_set(300)
    names = CMR_CLASSES
else:
    Xd, yd, table_d = X, y, echo_table
    names = ECHO_CLASSES
rows = np.arange(len(yd))
tr_d, te_d = train_test_split(rows, test_size=0.2, stratify=yd, random_state=RANDOM_STATE)
tr_d, va_d = train_test_split(tr_d, test_size=0.25, stratify=yd[tr_d], random_state=RANDOM_STATE)
mu, sigma = Xd[tr_d].mean(), Xd[tr_d].std()
T = lambda r: torch.tensor((Xd[r] - mu) / sigma, dtype=torch.float32).unsqueeze(1)
torch.manual_seed(RANDOM_STATE)
net = EchoCNN(n_classes=len(names), dropout=DROPOUT)
train_model(net, T(tr_d), torch.tensor(yd[tr_d]), T(va_d), torch.tensor(yd[va_d]), epochs=EPOCHS,
            augment_fn=augment if AUGMENT else None)
net.eval()
with torch.no_grad():
    pred_d = net(T(te_d)).argmax(dim=1).numpy()
print(pd.DataFrame(confusion_matrix(yd[te_d], pred_d), index=names, columns=names))
print("F1:", dict(zip(names, f1_score(yd[te_d], pred_d, average=None).round(3))))
print("macro F1:", round(f1_score(yd[te_d], pred_d, average="macro"), 3))

## Application challenges by field

| Field | Challenge |
|---|---|
| Heart failure | Give the echo frames three classes instead of two, normal, dilated and thick septum, from `echo_table`, and train `EchoCNN(n_classes=3)`. Which class does the network confuse most often, and with which? |
| Interventional cardiology | Train the network with 25, 50 and 100 percent of the training frames and plot the F1 of the abnormal class against the size of the training set. How many examples would a new catheterisation laboratory need to collect before a network like this is worth training? |
| Electrophysiology | Join `test_table` with the `ecg_rhythm` column of the registry and compare the share of missed abnormal frames in patients with atrial fibrillation and in sinus rhythm. Would you expect a difference on single frames, and on videos? |
| Cardiovascular imaging | Show the ten normal test frames with the highest probability of the abnormal class. What do they share, and which echocardiographic measurement would settle each case? |
| Preventive cardiology | Among the patients with hypertension, which share of the abnormal frames does the network miss, and how does this compare with the patients without hypertension? What does the answer mean for a screening programme of hypertensive heart disease? |
| Cardio-oncology | Follow one patient during chemotherapy: Draw frames of VC-0821 with `vclinic.imaging.echo_psax` while the diameter grows from 52 to 64 mm in steps of 2 mm, and plot the probability of the abnormal class of the course network against the diameter. Where does the network change its mind? |
| Intensive care | Time the prediction of the 400 test frames on your computer and compute the frames per second. Would the network keep up with a bedside echocardiogram at 50 frames per second? |

## After the notebook

Open the [interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-03/lab.html) and take the self-assessment in the tab *Check yourself*. Then write the reflection in the tab *Reflect and export* and export the learning log. The weekly project of Week 3, described on the [week page](README.md), trains the network of Section 5 on the slices of the cardiomyopathy clinic with the tools of Sections 6 and 7.

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>